# Notebook 27 — Full Game Simulation

## The Pokémon Company — PTCG AI Battle Challenge

### Team Jesus

Notebook 26 verified production-agent consistency, latency, accuracy, and runtime stability.

Notebook 27 advances from repeated single-state decisions to full-game simulation.

## Objectives

1. Inspect the existing battle-engine and integrated-agent interfaces.
2. Reuse the project’s battle-state and action models.
3. Create a full-game simulation controller.
4. Alternate turns between two agents.
5. apply legal actions to the evolving game state.
6. Detect game-ending conditions.
7. Record winners, turns, actions, errors, and termination reasons.
8. Run repeatable simulated matches.
9. Generate full-game statistics and reports.
10. Prepare the simulator for self-play and strategy optimization.

## Important distinction

A full game must include evolving states, alternating players, legal actions,
turn progression, victory conditions, and a final winner or termination result.

# Cell 2 — Imports

In [1]:
from __future__ import annotations

import importlib.util
import inspect
import json
import sys
import time
import types
import uuid

from dataclasses import asdict, dataclass, field, is_dataclass
from pathlib import Path
from typing import Any, Callable

print("Python:", sys.version)
print("Working directory:", Path.cwd())

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Working directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks


# Cell 3 — Locate the project and prior components

In [4]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    required_markers = {
        "notebooks",
        "scripts",
        "src",
        "reports",
    }

    for candidate in [current, *current.parents]:
        existing = {
            marker
            for marker in required_markers
            if (candidate / marker).exists()
        }

        if existing == required_markers:
            return candidate

    raise FileNotFoundError(
        "Could not locate the PTCG project root."
    )


PROJECT_ROOT = find_project_root()

BATTLE_ENGINE_EXPORT = (
    PROJECT_ROOT
    / "scripts"
    / "07_battle_engine.py"
)

INTEGRATED_AGENT_EXPORT = (
    PROJECT_ROOT
    / "scripts"
    / "12_integrated_battle_agent.py"
)

SELF_PLAY_EXPORT = (
    PROJECT_ROOT
    / "scripts"
    / "13_tournament_self_play_evaluation.py"
)

PRODUCTION_AGENT_EXPORT = (
    PROJECT_ROOT
    / "src"
    / "kaggle_agent"
    / "notebook21_export.py"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "notebook27"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SOURCE_FILES = {
    "battle_engine": BATTLE_ENGINE_EXPORT,
    "integrated_agent": INTEGRATED_AGENT_EXPORT,
    "self_play": SELF_PLAY_EXPORT,
    "production_agent": PRODUCTION_AGENT_EXPORT,
}

print("Project root:", PROJECT_ROOT)
print()

for name, path in SOURCE_FILES.items():
    print(
        f"{'[FOUND]' if path.is_file() else '[MISSING]'} "
        f"{name}: {path}"
    )

print()
print("Report directory:", REPORT_DIR)

missing_sources = [
    str(path)
    for path in SOURCE_FILES.values()
    if not path.is_file()
]

if missing_sources:
    raise FileNotFoundError(
        "Notebook 27 requires these files:\n"
        + "\n".join(missing_sources)
    )

print("\nAll required source files located.")

Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge

[FOUND] battle_engine: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\scripts\07_battle_engine.py
[FOUND] integrated_agent: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\scripts\12_integrated_battle_agent.py
[FOUND] self_play: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\scripts\13_tournament_self_play_evaluation.py
[FOUND] production_agent: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\kaggle_agent\notebook21_export.py

Report directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook27

All required source files located.


# Cell 4 — Safe export loader

In [5]:
def load_export_module(
    path: Path,
    label: str,
) -> types.ModuleType:
    source = path.read_text(
        encoding="utf-8-sig"
    )

    source_lines = [
        line
        for line in source.splitlines()
        if line.strip()
        != "from __future__ import annotations"
    ]

    cleaned_source = (
        "from __future__ import annotations\n"
        + "\n".join(source_lines)
    )

    module_name = (
        f"notebook27_{label}_"
        + uuid.uuid4().hex
    )

    module = types.ModuleType(module_name)
    module.__file__ = str(path)
    module.__package__ = ""

    sys.modules[module_name] = module

    compiled = compile(
        cleaned_source,
        str(path),
        "exec",
    )

    exec(
        compiled,
        module.__dict__,
    )

    return module


print("Safe export loader created.")

Safe export loader created.


# Cell 5 — Inspect declarations

In [7]:
import ast


def inspect_source_structure(
    path: Path,
) -> dict[str, list[str]]:
    source = path.read_text(
        encoding="utf-8-sig"
    )

    tree = ast.parse(
        source,
        filename=str(path),
    )

    classes = []
    functions = []
    imports = []
    assignments = []

    for node in ast.walk(tree):
        if isinstance(node, ast.ClassDef):
            classes.append(node.name)

        elif isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        ):
            functions.append(node.name)

        elif isinstance(node, ast.Import):
            for alias in node.names:
                imports.append(alias.name)

        elif isinstance(node, ast.ImportFrom):
            module_name = node.module or ""

            for alias in node.names:
                imports.append(
                    f"{module_name}.{alias.name}"
                )

        elif isinstance(
            node,
            (
                ast.Assign,
                ast.AnnAssign,
            ),
        ):
            targets = []

            if isinstance(node, ast.Assign):
                targets = node.targets
            else:
                targets = [node.target]

            for target in targets:
                if isinstance(target, ast.Name):
                    assignments.append(
                        target.id
                    )

    return {
        "classes": sorted(set(classes)),
        "functions": sorted(set(functions)),
        "imports": sorted(set(imports)),
        "assignments": sorted(set(assignments)),
    }


source_declarations = {}

for label, path in SOURCE_FILES.items():
    declarations = inspect_source_structure(
        path
    )

    source_declarations[label] = declarations

    print("=" * 72)
    print(label)
    print("=" * 72)

    print("Classes:")
    if declarations["classes"]:
        for name in declarations["classes"]:
            print(" -", name)
    else:
        print(" - None found")

    print("\nFunctions:")
    if declarations["functions"]:
        for name in declarations["functions"]:
            print(" -", name)
    else:
        print(" - None found")

    print("\nImports:")
    for name in declarations["imports"][:30]:
        print(" -", name)

    if len(declarations["imports"]) > 30:
        print(
            " - ...",
            len(declarations["imports"]) - 30,
            "more imports",
        )

    print("\nAssigned names:")
    for name in declarations["assignments"][:40]:
        print(" -", name)

    if len(declarations["assignments"]) > 40:
        print(
            " - ...",
            len(declarations["assignments"]) - 40,
            "more assigned names",
        )

    print()


battle_structure = source_declarations[
    "battle_engine"
]

assert (
    battle_structure["classes"]
    or battle_structure["functions"]
    or battle_structure["imports"]
), (
    "The battle-engine export appears to contain "
    "no detectable executable interface."
)

print("Source-interface inspection completed.")

battle_engine
Classes:
 - None found

Functions:
 - advance_turn
 - apply_damage
 - attach_energy_to_active
 - calculate_modified_damage
 - can_use_attack
 - check_winner
 - clean_type_value
 - count_attached_energy
 - create_battle_state
 - create_player_state
 - display_active_pokemon
 - display_turn_log
 - draw_cards
 - draw_valid_opening_hand
 - end_turn
 - first_card_by_category
 - get_base_damage
 - get_basic_pokemon
 - get_current_player
 - get_energy_cards_from_hand
 - get_opposing_player
 - get_usable_attacks
 - get_valid_attacks
 - handle_knockout
 - has_basic_pokemon
 - initialize_fresh_battle
 - initialize_player_from_deck
 - initialize_pokemon_state
 - load_pickle
 - perform_attack
 - play_simplified_turn
 - prepare_deck
 - promote_benched_pokemon
 - resolve_attack_action
 - run_battle_simulation
 - run_simulation_batch
 - safe_number
 - select_first_usable_attack
 - setup_starting_field
 - start_turn
 - summarize_attack_options
 - summarize_cards
 - summarize_player_state

# Cell 5A — Add project root to Python path

In [10]:
# Cell 5A — Make the project package importable

PROJECT_ROOT_STR = str(PROJECT_ROOT)

if PROJECT_ROOT_STR not in sys.path:
    sys.path.insert(0, PROJECT_ROOT_STR)

print("Project root added to sys.path:")
print(PROJECT_ROOT_STR)

print()
print("src exists:", (PROJECT_ROOT / "src").is_dir())
print("src __init__ exists:", (PROJECT_ROOT / "src" / "__init__.py").is_file())

assert (PROJECT_ROOT / "src").is_dir()

Project root added to sys.path:
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge

src exists: True
src __init__ exists: True


In [11]:
import src

print("src imported successfully.")
print("src location:", src.__file__)

src imported successfully.
src location: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\__init__.py


# Cell 6 — Import the existing full-game components

In [12]:
# Cell 6 — Import the existing full-game components

from src import (
    BattleSimulationResult,
    BattleState,
    BattleTurnRecord,
    PlayerState,
    PokemonBattleAgent,
    PokemonState,
    apply_move,
    create_battle_transcript,
    current_player_adapter,
    determine_battle_winner,
    evaluate_state_adapter,
    generate_moves_adapter,
    pokemon_state_features,
    simulate_ai_battle,
    terminal_state_adapter,
)

from src.engine.advanced_search import (
    AdvancedSearchEngine,
    ZobristHasher,
)

from src.tournament.runner import (
    run_two_agent_match,
)

print("Full-game components imported:")
print("- BattleState")
print("- PlayerState")
print("- PokemonState")
print("- PokemonBattleAgent")
print("- AdvancedSearchEngine")
print("- simulate_ai_battle")
print("- run_two_agent_match")
print("- determine_battle_winner")
print("- terminal_state_adapter")

print("\nExisting full-game interface loaded.")

Full-game components imported:
- BattleState
- PlayerState
- PokemonState
- PokemonBattleAgent
- AdvancedSearchEngine
- simulate_ai_battle
- run_two_agent_match
- determine_battle_winner
- terminal_state_adapter

Existing full-game interface loaded.


# Cell 7 — Inspect constructor and function signatures

In [13]:
# Cell 7 — Inspect constructor and function signatures

FULL_GAME_OBJECTS = {
    "BattleState": BattleState,
    "PlayerState": PlayerState,
    "PokemonState": PokemonState,
    "PokemonBattleAgent": PokemonBattleAgent,
    "AdvancedSearchEngine": AdvancedSearchEngine,
    "ZobristHasher": ZobristHasher,
    "simulate_ai_battle": simulate_ai_battle,
    "run_two_agent_match": run_two_agent_match,
    "apply_move": apply_move,
    "determine_battle_winner": determine_battle_winner,
    "terminal_state_adapter": terminal_state_adapter,
    "generate_moves_adapter": generate_moves_adapter,
}

for name, obj in FULL_GAME_OBJECTS.items():
    print("=" * 72)
    print(name)
    print("=" * 72)

    try:
        print("Signature:", inspect.signature(obj))
    except (TypeError, ValueError):
        print("Signature: unavailable")

    doc = inspect.getdoc(obj)

    if doc:
        print("Documentation:")
        for line in doc.splitlines()[:8]:
            print(" ", line)
    else:
        print("Documentation: none")

    print()

print("Full-game signatures inspected.")

BattleState
Signature: (player: 'PlayerState', opponent: 'PlayerState', turn_number: 'int', current_player: 'str') -> None
Documentation:
  Complete position used by minimax and advanced search.

PlayerState
Signature: (active: 'PokemonState', bench: 'List[PokemonState]', prize_cards_remaining: 'int' = 6, hand_size: 'int' = 7) -> None
Documentation:
  Represents one player's current battlefield.

PokemonState
Signature: (card: 'dict', current_hp: 'float', attached_energy: 'int' = 0, status: 'Optional[str]' = None, damage: 'float' = 0.0, is_active: 'bool' = False) -> None
Documentation:
  Represents one Pokémon currently in play.

PokemonBattleAgent
Signature: (engine: 'Any') -> 'None'
Documentation:
  Use an AdvancedSearchEngine to choose Pokémon actions.

AdvancedSearchEngine
Signature: (generate_moves: 'GenerateMovesFn', apply_move: 'ApplyMoveFn', evaluate_state: 'EvaluateFn', is_terminal: 'IsTerminalFn', current_player: 'CurrentPlayerFn', move_to_string: 'Optional[MoveToStringFn]' =

# Cell 8 — Inspect state-model fields

In [14]:
# Cell 8 — Inspect state-model fields

from dataclasses import MISSING, fields

STATE_MODELS = {
    "PokemonState": PokemonState,
    "PlayerState": PlayerState,
    "BattleState": BattleState,
}

for model_name, model_class in STATE_MODELS.items():
    print("=" * 72)
    print(model_name)
    print("=" * 72)

    if not is_dataclass(model_class):
        print("Not a dataclass.")
        print()
        continue

    for item in fields(model_class):
        if item.default is not MISSING:
            requirement = f"default={item.default!r}"
        elif item.default_factory is not MISSING:
            requirement = "default_factory"
        else:
            requirement = "required"

        print(
            f"- {item.name}: {item.type} "
            f"({requirement})"
        )

    print()

print("State-model inspection completed.")

PokemonState
- card: dict (required)
- current_hp: float (required)
- attached_energy: int (default=0)
- status: Optional[str] (default=None)
- damage: float (default=0.0)
- is_active: bool (default=False)

PlayerState
- active: PokemonState (required)
- bench: List[PokemonState] (required)
- prize_cards_remaining: int (default=6)
- hand_size: int (default=7)

BattleState
- player: PlayerState (required)
- opponent: PlayerState (required)
- turn_number: int (required)
- current_player: str (required)

State-model inspection completed.


# Cell 8 — Inspect state-model fields

In [15]:
# Cell 8 — Inspect state-model fields

from dataclasses import MISSING, fields

STATE_MODELS = {
    "PokemonState": PokemonState,
    "PlayerState": PlayerState,
    "BattleState": BattleState,
}

for model_name, model_class in STATE_MODELS.items():
    print("=" * 72)
    print(model_name)
    print("=" * 72)

    if not is_dataclass(model_class):
        print("Not a dataclass.")
        print()
        continue

    for item in fields(model_class):
        if item.default is not MISSING:
            requirement = f"default={item.default!r}"
        elif item.default_factory is not MISSING:
            requirement = "default_factory"
        else:
            requirement = "required"

        print(
            f"- {item.name}: {item.type} "
            f"({requirement})"
        )

    print()

print("State-model inspection completed.")

PokemonState
- card: dict (required)
- current_hp: float (required)
- attached_energy: int (default=0)
- status: Optional[str] (default=None)
- damage: float (default=0.0)
- is_active: bool (default=False)

PlayerState
- active: PokemonState (required)
- bench: List[PokemonState] (required)
- prize_cards_remaining: int (default=6)
- hand_size: int (default=7)

BattleState
- player: PlayerState (required)
- opponent: PlayerState (required)
- turn_number: int (required)
- current_player: str (required)

State-model inspection completed.


# Cell 9 — Inspect tournament classes and match runner

In [16]:
# Cell 9 — Inspect tournament interface

from src import (
    TournamentMatch,
    TournamentResult,
    TournamentStatistics,
)

TOURNAMENT_OBJECTS = {
    "TournamentMatch": TournamentMatch,
    "TournamentResult": TournamentResult,
    "TournamentStatistics": TournamentStatistics,
    "run_two_agent_match": run_two_agent_match,
}

for name, obj in TOURNAMENT_OBJECTS.items():
    print("=" * 72)
    print(name)
    print("=" * 72)

    try:
        print("Signature:", inspect.signature(obj))
    except (TypeError, ValueError):
        print("Signature: unavailable")

    if is_dataclass(obj):
        print("Fields:")

        for item in fields(obj):
            if item.default is not MISSING:
                requirement = f"default={item.default!r}"
            elif item.default_factory is not MISSING:
                requirement = "default_factory"
            else:
                requirement = "required"

            print(
                f"- {item.name}: {item.type} "
                f"({requirement})"
            )

    doc = inspect.getdoc(obj)

    if doc:
        print("Documentation:")
        for line in doc.splitlines()[:8]:
            print(" ", line)

    print()

print("Tournament interface inspected.")

TournamentMatch
Signature: (player_agent: 'PokemonBattleAgent', opponent_agent: 'PokemonBattleAgent', player_name: 'str' = 'Player', opponent_name: 'str' = 'Opponent', search_depth: 'int' = 6, max_turns: 'int' = 100) -> None
Fields:
- player_agent: PokemonBattleAgent (required)
- opponent_agent: PokemonBattleAgent (required)
- player_name: str (default='Player')
- opponent_name: str (default='Opponent')
- search_depth: int (default=6)
- max_turns: int (default=100)
Documentation:
  Configuration for one AI-versus-AI tournament match.

TournamentResult
Signature: (winner: 'str', turns: 'int', final_score: 'float', transcript: 'str') -> None
Fields:
- winner: str (required)
- turns: int (required)
- final_score: float (required)
- transcript: str (required)
Documentation:
  Summary of one completed tournament match.

TournamentStatistics
Signature: (matches_played: 'int' = 0, player_wins: 'int' = 0, opponent_wins: 'int' = 0, draws: 'int' = 0, total_turns: 'int' = 0, average_turns: 'float

# Cell 10 — Inspect the existing test-battle builder

In [17]:
# Cell 10 — Load the existing self-play helpers

self_play_module = load_export_module(
    SELF_PLAY_EXPORT,
    "self_play",
)

REQUIRED_SELF_PLAY_HELPERS = [
    "create_test_battle",
    "create_battle_agent",
    "run_match",
    "run_match_series",
]

missing_helpers = [
    name
    for name in REQUIRED_SELF_PLAY_HELPERS
    if not hasattr(self_play_module, name)
]

for name in REQUIRED_SELF_PLAY_HELPERS:
    print(
        f"{'[OK]' if hasattr(self_play_module, name) else '[MISSING]'} "
        f"{name}"
    )

if missing_helpers:
    raise AttributeError(
        "Self-play export is missing:\n"
        + "\n".join(missing_helpers)
    )

create_test_battle = self_play_module.create_test_battle
create_battle_agent = self_play_module.create_battle_agent
run_match = self_play_module.run_match
run_match_series = self_play_module.run_match_series

print()
print("create_test_battle signature:")
print(inspect.signature(create_test_battle))

print()
print("create_battle_agent signature:")
print(inspect.signature(create_battle_agent))

print()
print("run_match signature:")
print(inspect.signature(run_match))

print()
print("run_match_series signature:")
print(inspect.signature(run_match_series))

print("\nExisting self-play helpers loaded.")

✅ Notebook 13 imports completed.
Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
AdvancedSearchEngine         available
PokemonBattleAgent           available
BattleSimulationResult       available
BattleState                  available
PokemonState                 available
PlayerState                  available
ZobristHasher                available
simulate_ai_battle           available
generate_moves_adapter       available
evaluate_state_adapter       available
terminal_state_adapter       available
current_player_adapter       available
pokemon_state_features       available
apply_move                   available
✅ Tournament test cards created.
✅ Fresh tournament test battle created.
Player: Eevee ex
Opponent: Electrike
Starting side: Player
✅ Pokémon move key ready: ('Evolution Burst', 60.0, 2)
✅ Tournament battle agent created.
<class 'src.battle_agent.PokemonBattleAgent'>
<class 'src.engine.advanced_search.AdvancedSearchEngine'>
✅ Tournament agents c

# Cell 11 — Create fresh agents and the first Notebook 27 battle

In [18]:
# Cell 11 — Create fresh agents and a full-game battle

from src import TournamentMatch

player_agent = create_battle_agent(
    seed=20260727,
)

opponent_agent = create_battle_agent(
    seed=20260728,
)

initial_battle = create_test_battle(
    starting_player="Player",
)

full_game_match = TournamentMatch(
    player_agent=player_agent,
    opponent_agent=opponent_agent,
    player_name="Team Jesus AI",
    opponent_name="Baseline Opponent",
    search_depth=6,
    max_turns=100,
)

print("Player agent:", type(player_agent))
print("Opponent agent:", type(opponent_agent))
print("Starting side:", initial_battle.current_player)
print("Player active:", initial_battle.player.active.card.get("name"))
print("Opponent active:", initial_battle.opponent.active.card.get("name"))
print("Maximum turns:", full_game_match.max_turns)

assert player_agent is not opponent_agent
assert initial_battle.current_player == "Player"

print("\nFresh full-game match created.")

Player agent: <class 'src.battle_agent.PokemonBattleAgent'>
Opponent agent: <class 'src.battle_agent.PokemonBattleAgent'>
Starting side: Player
Player active: Eevee ex
Opponent active: Electrike
Maximum turns: 100

Fresh full-game match created.


# Cell 12 — Run the first true full-game match

In [19]:
# Cell 12 — Run the first true full-game match

started = time.perf_counter()

first_match_result = run_two_agent_match(
    full_game_match,
    initial_battle,
    verbose=True,
)

first_match_seconds = (
    time.perf_counter() - started
)

print()
print("=" * 72)
print("NOTEBOOK 27 — FIRST FULL-GAME RESULT")
print("=" * 72)

print("Winner:", first_match_result.winner)
print("Turns:", first_match_result.turns)
print("Final score:", first_match_result.final_score)
print("Elapsed seconds:", first_match_seconds)

assert first_match_result.winner in {
    "Team Jesus AI",
    "Baseline Opponent",
    "Draw",
}

assert 1 <= first_match_result.turns <= 100

print("\nFirst full-game simulation passed.")

Turn 1: Player — Eevee ex used Evolution Burst
  Player HP: 200.0 | Opponent HP: 10.0
Turn 2: Opponent — Electrike used Thunder Jolt
  Player HP: 170.0 | Opponent HP: 10.0
Turn 3: Player — Eevee ex used Evolution Burst
  Player HP: 170.0 | Opponent HP: 0.0

NOTEBOOK 27 — FIRST FULL-GAME RESULT
Winner: Team Jesus AI
Turns: 3
Final score: 170.0
Elapsed seconds: 0.0031450999958906323

First full-game simulation passed.


# Cell 13 — Inspect and save the match transcript

In [20]:
# Cell 13 — Inspect and save the match transcript

first_transcript = (
    first_match_result.transcript
    if hasattr(first_match_result, "transcript")
    else create_battle_transcript(first_match_result)
)

print(first_transcript)

TRANSCRIPT_FILE = (
    REPORT_DIR
    / "first_full_game_transcript.txt"
)

TRANSCRIPT_FILE.write_text(
    str(first_transcript),
    encoding="utf-8",
)

print()
print("Transcript saved:", TRANSCRIPT_FILE)

assert TRANSCRIPT_FILE.is_file()
assert TRANSCRIPT_FILE.stat().st_size > 0

print("\nFirst full-game transcript saved successfully.")

POKÉMON AI BATTLE TRANSCRIPT
Turn 1: Player — Eevee ex used Evolution Burst
  Damage: 60.0
  Search score: 633.00
  Search depth: 6
  Nodes searched: 17
  HP after move — Player: 200.0, Opponent: 10.0
  Next side: Opponent
----------------------------------------------------------------------
Turn 2: Opponent — Electrike used Thunder Jolt
  Damage: 30.0
  Search score: -633.00
  Search depth: 6
  Nodes searched: 10
  HP after move — Player: 170.0, Opponent: 10.0
  Next side: Player
----------------------------------------------------------------------
Turn 3: Player — Eevee ex used Evolution Burst
  Damage: 60.0
  Search score: 633.00
  Search depth: 6
  Nodes searched: 10
  HP after move — Player: 170.0, Opponent: 0.0
  Next side: Opponent
----------------------------------------------------------------------
Winner: Player
Stop reason: Battle reached a terminal state.

Transcript saved: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook27\first_full_game_tra

# Cell 14 — Run an alternating-start match series

In [21]:
# Cell 14 — Run a full-game match series

SERIES_MATCHES = 20

series_started = time.perf_counter()

series_results, series_statistics = run_match_series(
    full_game_match,
    number_of_matches=SERIES_MATCHES,
    alternate_starting_side=True,
)

series_elapsed_seconds = (
    time.perf_counter() - series_started
)

print("=" * 72)
print("NOTEBOOK 27 — FULL-GAME SERIES")
print("=" * 72)

print("Matches played:", series_statistics.matches_played)
print("Team Jesus AI wins:", series_statistics.player_wins)
print("Baseline Opponent wins:", series_statistics.opponent_wins)
print("Draws:", series_statistics.draws)
print("Average turns:", series_statistics.average_turns)
print("Average score:", series_statistics.average_score)
print("Elapsed seconds:", series_elapsed_seconds)

assert series_statistics.matches_played == SERIES_MATCHES

assert (
    series_statistics.player_wins
    + series_statistics.opponent_wins
    + series_statistics.draws
    == SERIES_MATCHES
)

print("\nFull-game series completed successfully.")

Match  1: start=Player   | winner=Team Jesus AI    | turns=3
Match  2: start=Opponent | winner=Team Jesus AI    | turns=4
Match  3: start=Player   | winner=Team Jesus AI    | turns=3
Match  4: start=Opponent | winner=Team Jesus AI    | turns=4
Match  5: start=Player   | winner=Team Jesus AI    | turns=3
Match  6: start=Opponent | winner=Team Jesus AI    | turns=4
Match  7: start=Player   | winner=Team Jesus AI    | turns=3
Match  8: start=Opponent | winner=Team Jesus AI    | turns=4
Match  9: start=Player   | winner=Team Jesus AI    | turns=3
Match 10: start=Opponent | winner=Team Jesus AI    | turns=4
Match 11: start=Player   | winner=Team Jesus AI    | turns=3
Match 12: start=Opponent | winner=Team Jesus AI    | turns=4
Match 13: start=Player   | winner=Team Jesus AI    | turns=3
Match 14: start=Opponent | winner=Team Jesus AI    | turns=4
Match 15: start=Player   | winner=Team Jesus AI    | turns=3
Match 16: start=Opponent | winner=Team Jesus AI    | turns=4
Match 17: start=Player  

# Cell 15 — Create the series results table

In [23]:
# Cell 15 — Create the full-game series DataFrame

import pandas as pd

series_rows = []

for match_number, result in enumerate(
    series_results,
    start=1,
):
    starting_side = (
        "Player"
        if match_number % 2 == 1
        else "Opponent"
    )

    series_rows.append(
        {
            "match_number": match_number,
            "starting_side": starting_side,
            "winner": result.winner,
            "turns": result.turns,
            "final_score": result.final_score,
            "transcript_characters": len(
                str(result.transcript)
            ),
        }
    )

series_df = pd.DataFrame(series_rows)

display(series_df)

print()
print("Winner counts:")
print(series_df["winner"].value_counts())

print()
print("Turn statistics:")
print(series_df["turns"].describe())

assert len(series_df) == SERIES_MATCHES
assert series_df["turns"].between(1, 100).all()

print("\nFull-game series table validated.")

,match_number,starting_side,winner,turns,final_score,transcript_characters
0,1,Player,Team Jesus AI,3,170.0,1024
1,2,Opponent,Team Jesus AI,4,140.0,1288
2,3,Player,Team Jesus AI,3,170.0,1024
3,4,Opponent,Team Jesus AI,4,140.0,1288
4,5,Player,Team Jesus AI,3,170.0,1024
5,6,Opponent,Team Jesus AI,4,140.0,1288
6,7,Player,Team Jesus AI,3,170.0,1024
7,8,Opponent,Team Jesus AI,4,140.0,1288
8,9,Player,Team Jesus AI,3,170.0,1024
9,10,Opponent,Team Jesus AI,4,140.0,1288



Winner counts:
winner
Team Jesus AI    20
Name: count, dtype: int64

Turn statistics:
count    20.000000
mean      3.500000
std       0.512989
min       3.000000
25%       3.000000
50%       3.500000
75%       4.000000
max       4.000000
Name: turns, dtype: float64

Full-game series table validated.


# Cell 16 — Save the full-game series reports

In [25]:
# Cell 16 — Save full-game simulation reports

SERIES_CSV = (
    REPORT_DIR
    / "full_game_series.csv"
)

SERIES_JSON = (
    REPORT_DIR
    / "full_game_summary.json"
)

series_df.to_csv(
    SERIES_CSV,
    index=False,
)

series_summary = {
    "project": "PTCG AI Battle Challenge",
    "team": "Team Jesus",
    "simulation_scope": (
        "Simplified full-game agent-versus-agent model"
    ),
    "matches_played": (
        series_statistics.matches_played
    ),
    "team_jesus_wins": (
        series_statistics.player_wins
    ),
    "baseline_wins": (
        series_statistics.opponent_wins
    ),
    "draws": series_statistics.draws,
    "average_turns": (
        series_statistics.average_turns
    ),
    "average_score": (
        series_statistics.average_score
    ),
    "elapsed_seconds": series_elapsed_seconds,
}

SERIES_JSON.write_text(
    json.dumps(
        series_summary,
        indent=4,
    ),
    encoding="utf-8",
)

print("Series CSV:", SERIES_CSV)
print("Summary JSON:", SERIES_JSON)

assert SERIES_CSV.is_file()
assert SERIES_JSON.is_file()
assert SERIES_CSV.stat().st_size > 0
assert SERIES_JSON.stat().st_size > 0

print("\nFull-game simulation reports saved.")

Series CSV: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook27\full_game_series.csv
Summary JSON: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook27\full_game_summary.json

Full-game simulation reports saved.


# Cell 17 — Validate starting-side fairness

In [26]:
# Cell 17 — Compare results by starting side

starting_side_summary = (
    series_df
    .groupby("starting_side")
    .agg(
        matches=("match_number", "count"),
        average_turns=("turns", "mean"),
        average_score=("final_score", "mean"),
        minimum_turns=("turns", "min"),
        maximum_turns=("turns", "max"),
    )
    .reset_index()
)

display(starting_side_summary)

winner_by_start = pd.crosstab(
    series_df["starting_side"],
    series_df["winner"],
)

print()
print("Winner distribution by starting side:")
display(winner_by_start)

assert starting_side_summary["matches"].sum() == SERIES_MATCHES
assert set(starting_side_summary["starting_side"]) == {
    "Player",
    "Opponent",
}

print("\nStarting-side comparison validated.")

,starting_side,matches,average_turns,average_score,minimum_turns,maximum_turns
0,Opponent,10,4.0,140.0,4,4
1,Player,10,3.0,170.0,3,3



Winner distribution by starting side:


winner,Team Jesus AI
starting_side,
Opponent,10
Player,10



Starting-side comparison validated.


# Cell 18 — Final Notebook 27 summary

In [28]:
# Cell 18 — Final Notebook 27 summary

matches_played = series_summary["matches_played"]

team_jesus_win_rate = series_summary.get(
    "team_jesus_win_rate",
    series_summary["team_jesus_wins"] / matches_played,
)

baseline_win_rate = series_summary.get(
    "baseline_win_rate",
    series_summary["baseline_wins"] / matches_played,
)

draw_rate = series_summary.get(
    "draw_rate",
    series_summary["draws"] / matches_played,
)

print("=" * 72)
print("Notebook 27 — Full Game Simulation")
print("=" * 72)
print()

print("Project:", series_summary["project"])
print("Team:", series_summary["team"])
print(
    "Simulation scope:",
    series_summary["simulation_scope"],
)

print()
print("First-match winner:", first_match_result.winner)
print("First-match turns:", first_match_result.turns)
print("First-match score:", first_match_result.final_score)

print()
print("Series matches:", matches_played)
print("Team Jesus AI wins:", series_summary["team_jesus_wins"])
print("Baseline wins:", series_summary["baseline_wins"])
print("Draws:", series_summary["draws"])
print("Team Jesus win rate:", f"{team_jesus_win_rate:.1%}")
print("Baseline win rate:", f"{baseline_win_rate:.1%}")
print("Draw rate:", f"{draw_rate:.1%}")
print("Average turns:", series_summary["average_turns"])
print("Average score:", series_summary["average_score"])

print()
print("Transcript:", TRANSCRIPT_FILE.name)
print("Series CSV:", SERIES_CSV.name)
print("Summary JSON:", SERIES_JSON.name)

assert matches_played == SERIES_MATCHES
assert (
    series_summary["team_jesus_wins"]
    + series_summary["baseline_wins"]
    + series_summary["draws"]
    == matches_played
)

print()
print("FULL-GAME SIMULATION VALIDATED")
print("NOTEBOOK 27 COMPLETED SUCCESSFULLY")

Notebook 27 — Full Game Simulation

Project: PTCG AI Battle Challenge
Team: Team Jesus
Simulation scope: Simplified full-game agent-versus-agent model

First-match winner: Team Jesus AI
First-match turns: 3
First-match score: 170.0

Series matches: 20
Team Jesus AI wins: 20
Baseline wins: 0
Draws: 0
Team Jesus win rate: 100.0%
Baseline win rate: 0.0%
Draw rate: 0.0%
Average turns: 3.5
Average score: 155.0

Transcript: first_full_game_transcript.txt
Series CSV: full_game_series.csv
Summary JSON: full_game_summary.json

FULL-GAME SIMULATION VALIDATED
NOTEBOOK 27 COMPLETED SUCCESSFULLY
